# Spotify Stream Count Extractor - Top 1000 Test
Extract Spotify stream counts for the top 1000 tracks (sorted by popularity) in `spotify_data.parquet`.
To speed up extraction, we use `spotify_scraper` to find the artist and then fetch the artist's total streaming data from `kworb.net`.

## 1. Install and Import Required Libraries

In [7]:
# %pip install spotifyscraper bs4 requests pandas pyarrow --quiet
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import os
from pathlib import Path

## 2. Load Dataset and Filter for Top 1000 Tracks

In [8]:
df = pd.read_parquet('spotify_411k.parquet')
print(f"Loaded {len(df)} tracks.")

df_test = df.sort_values(by='popularity', ascending=False).head(411000).reset_index(drop=True)
print(f"Processing top {len(df_test)} tracks by popularity.")

Loaded 411000 tracks.
Processing top 411000 tracks by popularity.


## 3. Configure Scraper & Kworb Helper

In [9]:
import spotify_scraper as sps
client = sps.SpotifyClient(log_level='WARNING')

def get_streams_kworb(track_id, track_name):
    try:
        # Fetch track info to get the primary artist ID
        info = client.get_track_info(f"https://open.spotify.com/track/{track_id}")
        if not info or 'artists' not in info or not info['artists']:
            return None
        
        artist_id = info['artists'][0]['id']
        
        # Fetch Kworb artist streaming page
        kworb_url = f"https://kworb.net/spotify/artist/{artist_id}_songs.html"
        resp = requests.get(kworb_url, timeout=5)
        if resp.status_code != 200:
            return None
        
        soup = BeautifulSoup(resp.text, 'html.parser')
        for tr in soup.find_all('tr'):
            cols = tr.find_all('td')
            if len(cols) >= 2:
                row_name = cols[0].get_text().strip()
                # Simple fuzzy match on track name
                if track_name.lower() in row_name.lower() or row_name.lower() in track_name.lower():
                    streams_str = cols[1].get_text().replace(',', '')
                    if streams_str.isdigit():
                        return int(streams_str)
        return None
    except Exception as e:
        print(f"Error for {track_name}: {e}")
        return None

## 4. Extract Streams in Batches of 100
Saves intermediate results to the `batch_results_streams` directory so we don't lose progress.

In [10]:
BATCH_SIZE = 100
BATCH_DIR = Path('batch_results_streams')
BATCH_DIR.mkdir(parents=True, exist_ok=True)

# --- RESUME LOGIC ---
processed_track_ids = set()
for f in BATCH_DIR.glob('results_batch_*.parquet'):
    try:
        chunk_df = pd.read_parquet(f, columns=['track_id'])
        processed_track_ids.update(chunk_df['track_id'].astype(str).tolist())
    except:
        pass

df_test['track_id'] = df_test['track_id'].astype(str)
remaining_tracks = df_test[~df_test['track_id'].isin(processed_track_ids)].reset_index(drop=True)
print(f"Found {len(processed_track_ids)} already processed tracks.")
print(f"Remaining tracks to process: {len(remaining_tracks)}")
# --------------------

new_results = []
total_processed = 0

# Instead of fetching everything at once, we chunk it into BATCH_SIZE
for batch_idx in range(0, len(remaining_tracks), BATCH_SIZE):
    batch_df = remaining_tracks.iloc[batch_idx:batch_idx+BATCH_SIZE]
    batch_results = []
    
    for _, row in batch_df.iterrows():
        tid = row['track_id']
        tname = row['track_name']
        
        streams = get_streams_kworb(tid, tname)
        batch_results.append({'track_id': tid, 'stream_count': streams})
        print(f"{tname[:40].ljust(40)} -> {streams}")
        
        time.sleep(0.01) # Be polite to Kworb
        
    # Save batch to disk
    if batch_results:
        batch_out_df = pd.DataFrame(batch_results)
        # Calculate correct batch index based on existing files to avoid overwriting
        existing_files = len(list(BATCH_DIR.glob('results_batch_*.parquet')))
        batch_file = BATCH_DIR / f'results_batch_{existing_files + 1:04d}.parquet'
        batch_out_df.to_parquet(batch_file, index=False)
        print(f"--- Saved {len(batch_out_df)} records to {batch_file} ---")
    
    new_results.extend(batch_results)
    total_processed += len(batch_results)
    
print(f"\nProcessing complete. Extracted {total_processed} new tracks.")


Found 406900 already processed tracks.
Remaining tracks to process: 4100
Midnight Rhythm                          -> None
Us Or Them                               -> None
Malvon                                   -> None
Argentinian Folk Song in F-Sharp Minor   -> None
My Girl                                  -> None
Sticky Situation - General Dub Mix       -> None
Worlds Apart                             -> None
Darkest Love                             -> None
Wind of Doom                             -> None
El Perrito Viejo                         -> None
The Grey - Original Mix                  -> None
Rusty Circles - Live                     -> None
This Damned Love                         -> None
Big Girls - Original Mix                 -> None
Ai Nós Bebe                              -> None
Quakers                                  -> None
No Glory - Live                          -> None
Not This Time - Original Mix             -> None
You Want Me Back - Paolo Madzone Zampett -> N

## 5. Combine Batches & Final Output

In [11]:
combined_df = pd.DataFrame(new_results)
# Or just load everything from BATCH_DIR if resuming from a broken state
all_files = sorted(BATCH_DIR.glob('results_batch_*.parquet'))
if not combined_df.empty or all_files:
    temp_dfs = [pd.read_parquet(f) for f in all_files]
    final_df = pd.concat(temp_dfs, ignore_index=True)
    
    final_df.to_parquet('spotify_stream_counts.parquet', index=False)
    print(f"Saved {len(final_df)} total stream counts to 'spotify_stream_counts.parquet'")
else:
    print("No data to save.")

Saved 428800 total stream counts to 'spotify_stream_counts.parquet'


/tmp/ipykernel_3268030/1319970653.py:6: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_df = pd.concat(temp_dfs, ignore_index=True)


## 6. Examination of Stream Counts Parquet
Load the saved parquet file and show the first 10 lines.

In [12]:
import pandas as pd
df_streams = pd.read_parquet('batch_results_streams/results_batch_0001.parquet')
df_streams.head(10)

,track_id,stream_count
0,0yLdNVWF3Srea0uzk55zFn,2.809638e+09
1,4nrPB8O7Y7wsOCJdgXkthe,1.153606e+09
2,7oDd86yk8itslrA9HRP2ki,3.189438e+09
3,1Qrg8KqiBpW07V7PNxwwwL,2.747259e+09
4,5ww2BF9slyYgNOk37BlC4u,2.374237e+09
5,4uUG5RXrOk84mYEfFvj3cK,2.343735e+09
6,0WtM2NBVQNNJLh6scP13H8,1.839255e+09
7,65FftemJ1DbbZ45DUfHJXE,9.196280e+08
8,2tTmW7RDtMQtBk7m2rYeSw,2.047428e+09
9,78Sw5GDo6AlGwTwanjXbGh,1.753805e+09
